# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What real estate transaction in Pittsburgh was more than $500,000**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-10-30 13:18:04 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-10-30T13:18:04.430342")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'real estate sales transactions Pittsburgh property':

1. **City-Owned Property (Real Estate Tax Database)** (55.8% match)
   Resource: City-Owned Properties
   Resource ID: `4ff5eb17-e2ad-4818-97c4-8f91fc6b6396`
   Dataset ID: `city-owned-property`
   Format: CSV | Rows: 5,108 | Cols: 19
   Tags: property_sales, real_estate, geographic_location, tax_records, sale_price, property_identification, municipal_districts, neighborhood_data, payment_history, treasury_sales
   This dataset contains real estate transaction information, including property ide
```


In [ ]:
# Step 1: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": 'real estate sales transactions Pittsburgh property', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `real estate sales`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 5 datasets matching 'real estate sales'

1. **City-Owned Property (Real Estate Tax Database)**
   ID: `city-owned-property`
   This dataset contains a listing of property owned by the City of Pittsburgh obtained from the City's Real Estate Database. For a more complete listing of City-owned properties obta
   - City-Owned Properties (CSV) [DataStore] ID: `4ff5eb17-e2ad-4818-97c4-8f91fc6b6396`
   - Data Dictionary with Field Defintions (XLSX) [DataStore] ID: `9070809b-54d9-441d-802e-812fcb7c825a`

2. **City Treasury Sales**
   ID: `city-treasury-sales`
   A listing of all the properties c
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'real estate sales', "rows": 10, "fq": 'organization:city-of-pittsburgh'}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'Allegheny County property sale transactions sale price deed':

1. **City Treasury Sales** (47.8% match)
   Resource: City Treasury Sales
   Resource ID: `6b2aa631-26e0-4d02-abe0-7fb87707210c`
   Dataset ID: `city-treasury-sales`
   Format: CSV | Rows: 48 | Cols: 24
   Tags: _etl, delinquency, finance, property, real estate, sale, treasury
   This dataset provides comprehensive property information, including identification numbers (PIN), addresses, and tax assessments. It details various tax components such as city, county, school, librar
   Covera
```


In [ ]:
# Step 3: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": 'Allegheny County property sale transactions sale price deed', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 4: Search for Datasets

**Search query:** `property sale transactions Allegheny County`

**Result preview:**
```
Found 8 datasets matching 'property sale transactions Allegheny County'

1. **Allegheny County Property Sale Transactions**
   ID: `real-estate-sales`
   This dataset contains data on all Real Property parcels that have sold since 2013 in Allegheny County, PA.  Before doing any market analysis on property sales, check the sales va
   - Property Sales Transactions (CSV) [DataStore] ID: `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
   - Allegheny County Information Portal Real Estate Sales (HTML) ID: `bfdf3a78-7830-425d-afa2-8fbc99c79dcb`
   - Parcels N'at (HTML) ID: `637ab4db-1313-4512-9c2b-6fa65975
```


In [ ]:
# Step 4: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'property sale transactions Allegheny County', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 5: Dataset Details

**Dataset:** `real-estate-sales`

**Result preview:**
```
# Allegheny County Property Sale Transactions

This dataset contains data on all Real Property parcels that have sold since 2013 in Allegheny County, PA.

Before doing any market analysis on property sales, check the sales validation codes. Many property "sales" are not considered a valid representation of the true market value of the property. For example, when multiple lots are together on one deed with one price they are generally coded as invalid ("H") because the sale price for each parcel ID number indicates the total price paid for a group of parcels, not just for one parcel. See the 
```


In [ ]:
# Step 5: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'real-estate-sales'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 6: Load Data from Resource

**Resource ID:** `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
**Limit:** 5

**Result preview:**
```
Resource: 5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1
Total records: 506,991
Loaded: 5
Fields (25): PARID, FULL_ADDRESS, PROPERTYHOUSENUM, PROPERTYFRACTION, PROPERTYADDRESSDIR, PROPERTYADDRESSSTREET, PROPERTYADDRESSSUF, PROPERTYADDRESSUNITDESC, PROPERTYUNITNO, PROPERTYCITY, PROPERTYSTATE, PROPERTYZIP, SCHOOLCODE, SCHOOLDESC, MUNICODE, MUNIDESC, RECORDDATE, SALEDATE, PRICE, DEEDBOOK, DEEDPAGE, SALECODE, SALEDESC, INSTRTYP, INSTRTYPDESC

Sample (5 rows):

           PARID                          FULL_ADDRESS  PROPERTYHOUSENUM PROPERTYFRACTION PROPERTYADDRESSDIR PROPERTYADDRESSSTREET PROPERTYADDRESSSUF
```


In [ ]:
# Step 6: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 7: SQL Analysis Query

**SQL:**
```sql
SELECT "SALECODE", "SALEDESC", COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' GROUP BY "SALECODE","SALEDESC" ORDER BY cnt DESC LIMIT 20
```

**Result preview:**
```
SQL: SELECT "SALECODE", "SALEDESC", COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' GROUP BY "SALECODE","SALEDESC" ORDER BY cnt DESC LIMIT 20
Rows: 20
Columns: SALECODE, SALEDESC, cnt

SALECODE                                SALEDESC   cnt
       3                 LOVE AND AFFECTION SALE 54892
       0                              VALID SALE 54004
       H                       MULTI-PARCEL SALE 34903
      14          TIME ON MARKET (INSUFF/EXCESS) 24769
      36                              QUIT CLAIM 23771
       9           OTHER INVALID SALE
```


In [ ]:
# Step 7: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "SALECODE", "SALEDESC", COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' GROUP BY "SALECODE","SALEDESC" ORDER BY cnt DESC LIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 8: SQL Analysis Query

**SQL:**
```sql
SELECT COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND CAST("PRICE" AS FLOAT) > 500000
```

**Result preview:**
```
SQL: SELECT COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND CAST("PRICE" AS FLOAT) > 500000 LIMIT 10000
Rows: 1
Columns: cnt

  cnt
16523
```


In [ ]:
# Step 8: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' AND CAST("PRICE" AS FLOAT) > 500000'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 9: SQL Analysis Query

**SQL:**
```sql
SELECT "FULL_ADDRESS", "SALEDATE", "PRICE", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND "SALECODE" = '0' AND CAST("PRICE" AS FLOAT) > 500000 ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10
```

**Result preview:**
```
SQL: SELECT "FULL_ADDRESS", "SALEDATE", "PRICE", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND "SALECODE" = '0' AND CAST("PRICE" AS FLOAT) > 500000 ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10
Rows: 10
Columns: FULL_ADDRESS, SALEDATE, PRICE, SALEDESC

                                FULL_ADDRESS   SALEDATE       PRICE   SALEDESC
      1001 LIBERTY AVE, PITTSBURGH, PA 15222 2013-08-09 124400000.0 VALID SALE
   525 WILLIAM PENN PL, PITTSBURGH, PA 15219 2016-08-10  67650000.0 VALID SALE
           436 7TH AVE, PITTSBURGH, PA 15219 2013-11-26  1724
```


In [ ]:
# Step 9: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "FULL_ADDRESS", "SALEDATE", "PRICE", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' AND "SALECODE" = \'0\' AND CAST("PRICE" AS FLOAT) > 500000 ORDER BY CAST("PRICE" AS FLOAT) DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 10: SQL Analysis Query

**SQL:**
```sql
SELECT COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND "SALECODE" = '0' AND CAST("PRICE" AS FLOAT) > 500000
```

**Result preview:**
```
SQL: SELECT COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND "SALECODE" = '0' AND CAST("PRICE" AS FLOAT) > 500000 LIMIT 10000
Rows: 1
Columns: cnt

 cnt
4845
```


In [ ]:
# Step 10: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT COUNT(*) as cnt FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' AND "SALECODE" = \'0\' AND CAST("PRICE" AS FLOAT) > 500000'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 11: SQL Analysis Query

**SQL:**
```sql
SELECT "FULL_ADDRESS", "SALEDATE", "PRICE", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND "SALECODE" = '0' AND CAST("PRICE" AS FLOAT) > 500000 ORDER BY "SALEDATE" DESC LIMIT 10
```

**Result preview:**
```
SQL: SELECT "FULL_ADDRESS", "SALEDATE", "PRICE", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = 'PITTSBURGH' AND "SALECODE" = '0' AND CAST("PRICE" AS FLOAT) > 500000 ORDER BY "SALEDATE" DESC LIMIT 10
Rows: 10
Columns: FULL_ADDRESS, SALEDATE, PRICE, SALEDESC

                                     FULL_ADDRESS   SALEDATE    PRICE   SALEDESC
           624 EAST END AVE, PITTSBURGH, PA 15221 2026-09-18 525000.0 VALID SALE
           140 WOODHAVEN DR, PITTSBURGH, PA 15228 2026-09-18 760000.0 VALID SALE
111 GRANDVIEW AVE, UNIT 703, PITTSBURGH, PA 15211 2026-09-18 890000
```


In [ ]:
# Step 11: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "FULL_ADDRESS", "SALEDATE", "PRICE", "SALEDESC" FROM "5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1" WHERE "PROPERTYCITY" = \'PITTSBURGH\' AND "SALECODE" = \'0\' AND CAST("PRICE" AS FLOAT) > 500000 ORDER BY "SALEDATE" DESC LIMIT 10'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer (as delivered in chat)

> **## Real Estate Transactions in Pittsburgh Over $500,000

Using the **Allegheny County Property Sale Transactions** dataset (WPRDC portal, covers sales since 2012), there are **4,845 valid ("VALID SALE") property transactions** in Pittsburgh with a sale price over $500,000 — and 16,523 if you include all sale types (including multi-parcel, foreclosure, corrective deeds, etc., which don't reflect true market value).

### Top 5 largest Pittsburgh real estate transactions on record (valid sales):
| Address | Sale Date | Price |
|---|---|---|
| 1001 Liberty Ave, Pittsburgh, PA 15222 | 2013-08-09 | **$124,400,000** |
| 525 William Penn Pl, Pittsburgh, PA 15219 | 2016-08-10 | **$67,650,000** |
| 436 7th Ave, Pittsburgh, PA 15219 | 2013-11-26 | **$17,241,000** |
| 610 Epsilon Dr, Pittsburgh, PA 15238 | 2019-01-23 | **$8,500,000** |
| 717-745 Grant St, Pittsburgh, PA 15219 | 2013-11-21 | **$6,980,000** |

### Recent examples just above $500,000 (most recent transactions in the dataset, Sept 2026):
- 624 East End Ave — $525,000
- 1201 Ingomar Heights Rd — $516,700
- 459 Troutwood Dr — $552,456
- 2359 James Dr — $642,500
- 6313 Kentucky Ave — $630,000

**Source:** "Allegheny County Property Sale Transactions" dataset on WPRDC (data.wprdc.org), resource "Property Sales Transactions," which covers all recorded Real Property parcel sales in Allegheny County from 2012 to present.

**Caveat:** The dataset includes a `SALECODE` field that flags whether a sale is a reliable indicator of market value. I filtered to `SALECODE = '0'` ("VALID SALE") for the figures above; many other recorded transactions above $500,000 exist but are coded as multi-parcel sales, sheriff sales, corrective deeds, etc., which don't represent true market-value transactions for a single property.**

*The figures above restate the assistant's answer for convenience — they are
not computed by this cell. The derivation lives in the code cells earlier in
this notebook; re-run them to re-derive every number from the source APIs.*

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Western PA Regional Data Center (WPRDC)
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-10-30

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-10-30 13:18:04
- **Query**: What real estate transaction in Pittsburgh was more than $500,000
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
